In [2]:
# Test Lernen der Dynamik
import numpy as np
from NN import Neural_ODE_MLP

# --- 1. Daten inspizieren ---
raw = np.loadtxt("OCP_dynamic_data.csv", delimiter=",", skiprows=0)
M = int(raw[:, 0].max()) + 1
N = raw.shape[0] // M
print(f"CSV shape: {raw.shape}  → M={M}, N={N}")
print(f"t range:   [{raw[:,1].min():.3f}, {raw[:,1].max():.3f}]")
print(f"x range:   [{raw[:,2].min():.3f}, {raw[:,2].max():.3f}]")
print(f"u range:   [{raw[:,3].min():.3f}, {raw[:,3].max():.3f}]")
print(f"dx/dt range: [{(raw[:,2].max()-raw[:,2].min()):.3f}]")

# --- 2. Modell ---
model = Neural_ODE_MLP(layers=20, dim_state=1, dim_control=1, method='sensitivity')
model.read_data("OCP_dynamic_data.csv")

# --- 3. Training ---
lr = 1e-2
for epoch in range(2000):
    loss, lr = model.train_step(learning_rate_init=lr, max_iter=15)
    if epoch % 200 == 0:
        print(f"epoch {epoch:4d}  loss={loss:.3e}  lr={lr:.2e}")

# --- 4. Punktweise Validierung ---
print("\nValidierung f(x,u) = x + u + 1:")
for xt, ut in [(0.0, 0.0), (0.5, 0.3), (-1.0, 1.0), (1.0, -0.5)]:
    f_l  = model.f_point(xt, ut)[0]
    fx_l = model.dfdx(xt, ut)[0,0]
    fu_l = model.dfdu(xt, ut)[0,0]
    print(f"  (x={xt:+.1f}, u={ut:+.1f}): "
          f"f={f_l:.4f} (true {xt+ut+1:.4f}),  "
          f"f_x={fx_l:.4f} (1.0),  f_u={fu_l:.4f} (1.0)")

# --- 5. Vorwärtssimulation: gelernt vs. wahr ---
x0 = 0.0
u_func = lambda t: 0.5*np.sin(2*t)
t_sim = np.linspace(0, model.time_grid[-1], 200)
dt_sim = t_sim[1] - t_sim[0]

x_true  = np.zeros(len(t_sim)); x_true[0]  = x0
x_learn = np.zeros(len(t_sim)); x_learn[0] = x0
for i in range(len(t_sim)-1):
    u = u_func(t_sim[i])
    x_true[i+1]  = x_true[i]  + dt_sim*(x_true[i]  + u + 1)
    x_learn[i+1] = x_learn[i] + dt_sim*model.f_point(x_learn[i], u)[0]

print(f"\nSimulation über [{t_sim[0]:.2f}, {t_sim[-1]:.2f}]:")
print(f"  max|x_true - x_learned| = {np.max(np.abs(x_true-x_learn)):.3e}")

CSV shape: (100000, 4)  → M=200, N=500
t range:   [0.000, 1.000]
x range:   [-15.587, 13.232]
u range:   [-7.959, 6.218]
dx/dt range: [28.819]
epoch    0  loss=9.656e+02  lr=2.75e-03


KeyboardInterrupt: 